# Resident Reintegration Readiness Scorer

## 1. Problem framing
Case managers need **prioritization** for residents approaching reintegration—who may need more visits, planning, or resources in the next window. **Stakeholders:** social workers, clinical supervisor, program director. **Predictive vs explanatory:** **Primary predictive**—out-of-sample discrimination for a binary near-term outcome; **secondary explanatory**—logistic-style interpretation where exported for transparency. **Justification:** Operations require **ranking** under capacity constraints (prediction); governance requires **interpretable** drivers (explanation)—textbook dual use.

## 2. Data acquisition, preparation & exploration
Residents joined to **process_recordings**, **home_visitations**, **education_records**, **health_wellbeing_records**, etc., at observation dates with **no future leakage**. Modules: `data_prep.py`, `feature_engineering.py`. Notebook steps profile counts/missingness. **Reproducible pipeline:** `run_all.py`, `export_artifacts.py`, `score_all_current_residents.py`.

## 3. Modeling & feature selection
`train_predictive` / `train_explanatory` and final calibration in `model_finalize.py` as applicable. Features encode engagement intensity, risk flags, and trajectory summaries. Hyperparameters and CV in training scripts.

## 4. Evaluation & interpretation
ROC/PR, calibration, Brier where applicable; stratified splits. **False negative** (miss a struggling resident) risks **poor transition**; **false positive** (over-alert) consumes **scarce staff time**—asymmetric costs favor recall for high-risk bands with human triage.

## 5. Causal and relationship analysis
Importances show **which recorded factors co-occur** with modeled readiness—not proof that visits *cause* success (reverse causality, selection into services). Discuss limitations honestly; use scores as **decision support** only.

## 6. Deployment notes
**Artifacts:** `serialized_models/` + `outputs/*.json`. **Backend:** `refresh_ml_artifacts.py --reintegration-only` → `App_Data/ml/reintegration/`. **.NET:** `MlResidentsController` and related services read JSON. **UI:** `ResidentsListPage.tsx` ML columns; `AdminHomePage` widget; `frontend/src/lib/mlApi.ts` (`getResidentPriority`, `getResidentCurrentScores`).

In [16]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name in ("ml_pipeline", "ml-pipelines"):
    ML_PIPELINE = NOTEBOOK_DIR
elif NOTEBOOK_DIR.name == "ml_pipeline_reintegration_readiness_scorer":
    ML_PIPELINE = NOTEBOOK_DIR.parent
else:
    ML_PIPELINE = NOTEBOOK_DIR / "WinterIntex4-5" / "ml-pipelines"

if str(ML_PIPELINE) not in sys.path:
    sys.path.insert(0, str(ML_PIPELINE))

DATA_DIR = ML_PIPELINE.parent / "data" / "lighthouse_csv_v7"
print("ML_PIPELINE:", ML_PIPELINE)
print("DATA_DIR:", DATA_DIR)

ML_PIPELINE: /Users/jooyoung/Downloads/intex2026/WinterIntex4-5/ml_pipeline
DATA_DIR: /Users/jooyoung/Downloads/intex2026/WinterIntex4-5/data/lighthouse_csv_v7


## Step 1 — Load & profile tables

In [17]:
from ml_pipeline_reintegration_readiness_scorer.data_prep import load_tables, parse_datetime_columns, profile_tables

tables = parse_datetime_columns(load_tables(DATA_DIR))
prof = profile_tables(tables)
for name, p in prof.items():
    print(f"{name}: rows={p['rows']}")
    print("  missing (top):", list(p["missing_top"].items())[:5])

residents: rows=60
  missing (top): [('notes_restricted', 1.0), ('pwd_type', 0.95), ('special_needs_diagnosis', 0.9), ('date_closed', 0.5), ('referring_agency_person', 0.4)]
process_recordings: rows=2819
  missing (top): [('notes_restricted', 1.0), ('recording_id', 0.0), ('resident_id', 0.0), ('session_date', 0.0), ('social_worker', 0.0)]
home_visitations: rows=1337
  missing (top): [('follow_up_notes', 0.4106207928197457), ('family_members_present', 0.31787584143605085), ('visitation_id', 0.0), ('resident_id', 0.0), ('visit_date', 0.0)]
education_records: rows=534
  missing (top): [('education_record_id', 0.0), ('resident_id', 0.0), ('record_date', 0.0), ('education_level', 0.0), ('school_name', 0.0)]
health_wellbeing_records: rows=534
  missing (top): [('health_record_id', 0.0), ('resident_id', 0.0), ('record_date', 0.0), ('general_health_score', 0.0), ('nutrition_score', 0.0)]
incident_reports: rows=100
  missing (top): [('resolution_date', 0.29), ('incident_id', 0.0), ('resident_id

## Step 2–3 — Snapshot dataset & label

In [18]:
from ml_pipeline_reintegration_readiness_scorer.feature_engineering import build_snapshot_frame, global_data_cutoff

print("Data cutoff:", global_data_cutoff(tables))
df, meta = build_snapshot_frame(tables)
print(json.dumps(meta, indent=2))
display(df[df.columns[:8]].head())
print(df["reintegration_success_next_60_days"].value_counts())

Data cutoff: 2027-02-02 00:00:00
{
  "n_snapshots": 1609,
  "n_residents": 60,
  "data_cutoff": "2027-02-02 00:00:00",
  "success_horizon_days": 60,
  "positive_rate": 0.014916096954630205
}


,pr_sessions_total,pr_sessions_last_30d,pr_mean_duration,pr_progress_rate,pr_concern_rate,pr_mean_gap_days,pr_emotional_improvement_trend,hv_total_visits
0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,2.0,2.0,72.500000,1.000000,1.000000,3.000000,-0.500000,2.0
2,8.0,6.0,73.125000,0.875000,0.250000,6.142857,-0.041738,4.0
3,11.0,3.0,74.727273,0.909091,0.272727,5.600000,-0.005102,9.0
4,15.0,4.0,65.933333,0.866667,0.200000,7.071429,0.010684,10.0


reintegration_success_next_60_days
0    1585
1      24
Name: count, dtype: int64


## Step 6–8 — Train, evaluate, coefficients

In [19]:
from ml_pipeline_reintegration_readiness_scorer.export_artifacts import export_all

info = export_all(data_dir=DATA_DIR)
print(json.dumps(info, indent=2, default=str))

{
  "rows": 1609,
  "best_model": "random_forest",
  "selected_threshold": 0.13,
  "holdout_roc_auc": 0.9089775561097256,
  "holdout_pr_auc": 0.03835227272727273,
  "brier_holdout": 0.006301015517030328,
  "calibration_used": false,
  "metrics_at_0.5": {
    "accuracy": 0.9950372208436724,
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "positive_rate_pred": 0.0,
    "brier_score": 0.006301015517030328,
    "roc_auc": 0.9089775561097256,
    "pr_auc": 0.03835227272727273,
    "confusion_matrix": [
      [
        401,
        0
      ],
      [
        2,
        0
      ]
    ],
    "top_10pct_triage": {
      "precision_at_top_k": 0.024390243902439025,
      "k_n": 41,
      "k_frac": 0.1,
      "recall_at_top_k": 0.5,
      "positives_in_top_k": 1,
      "positives_total": 2
    }
  },
  "metrics_at_tuned": {
    "accuracy": 0.9553349875930521,
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "positive_rate_pred": 0.03970223325062035,
    "brier_score": 0.0063

In [20]:
from ml_pipeline_reintegration_readiness_scorer import config

with open(config.SERIALIZED_DIR / "reintegration_readiness_metadata.json", encoding="utf-8") as f:
    m = json.load(f)
print("Best model:", m["best_model"])
print("Selected threshold (OOF-tuned):", m.get("selected_threshold"))
# Metadata now reports holdout metrics at default 0.5 and at the tuned threshold (see README)
print("Holdout @ threshold 0.5:\n", json.dumps(m.get("holdout_metrics_threshold_0.5"), indent=2, default=str))
print("Holdout @ tuned threshold:\n", json.dumps(m.get("holdout_metrics_tuned_threshold"), indent=2, default=str))
coef = pd.read_csv(config.OUTPUTS_DIR / "explanatory_logistic_coefficients.csv")
display(coef.head(15))

Best model: random_forest
Selected threshold (OOF-tuned): 0.13
Holdout @ threshold 0.5:
 {
  "accuracy": 0.9950372208436724,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "positive_rate_pred": 0.0,
  "brier_score": 0.006301015517030328,
  "roc_auc": 0.9089775561097256,
  "pr_auc": 0.03835227272727273,
  "confusion_matrix": [
    [
      401,
      0
    ],
    [
      2,
      0
    ]
  ],
  "top_10pct_triage": {
    "precision_at_top_k": 0.024390243902439025,
    "k_n": 41,
    "k_frac": 0.1,
    "recall_at_top_k": 0.5,
    "positives_in_top_k": 1,
    "positives_total": 2
  }
}
Holdout @ tuned threshold:
 {
  "accuracy": 0.9553349875930521,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "positive_rate_pred": 0.03970223325062035,
  "brier_score": 0.006301015517030328,
  "roc_auc": 0.9089775561097256,
  "pr_auc": 0.03835227272727273,
  "confusion_matrix": [
    [
      385,
      16
    ],
    [
      2,
      0
    ]
  ],
  "threshold": 0.13,
  "top_10pct_triage": {
    "pr

,feature,coef,abs_coef
0,intercept,-6.554781,6.554781
1,cat__case_category_Neglected,-3.493098,3.493098
2,cat__case_category_Surrendered,2.821109,2.821109
3,num__edu_progress_trend,-2.556057,2.556057
4,cat__case_category_Abandoned,2.209109,2.209109
5,num__hv_total_visits,1.862977,1.862977
6,num__pr_concern_rate,1.678512,1.678512
7,cat__case_category_Foundling,-1.583695,1.583695
8,num__days_since_enrollment,-1.414332,1.414332
9,num__hw_mean_general_health,1.195788,1.195788
